# Hamilton's Equations of Motion

This notebook contains the programmatic verification for the **Hamilton's Equations of Motion** entry from the THEORIA dataset.

**Entry ID:** hamiltons_equations  
**Required Library:** sympy 1.13.1

## Description
Hamilton's equations reformulate classical mechanics using generalized coordinates and conjugate momenta as independent variables. They transform second-order Euler-Lagrange equations into first-order differential equations, providing a powerful framework for analyzing dynamical systems. This formulation bridges to quantum mechanics through canonical quantization and is essential for phase space analysis and conservation laws.

## Installation
First, let's install the required library:

In [ ]:
# Install required library with exact version
!pip install sympy==1.13.1

## Programmatic Verification

The following code verifies the derivation mathematically:

In [ ]:
import sympy as sp

# =====================================================
# Programmatic verification: Hamilton's Equations
# Verifying each derivation step explicitly
# Note: We verify for a single degree of freedom (i=1).
# The derivation is index-by-index, so verification for
# one index i suffices to establish the general result.
# =====================================================

# Define symbols
t = sp.Symbol('t', real=True)
q_i, p_i = sp.symbols('q_i p_i', real=True)
q_dot_i = sp.Symbol('q_dot_i', real=True)

# Abstract functions
L = sp.Function('L')
H = sp.Function('H')

# Symbols for partial derivatives
dL_dq = sp.Symbol('dL_dq', real=True)  # ∂L/∂q_i
dL_dqdot = sp.Symbol('dL_dqdot', real=True)  # ∂L/∂q_dot_i
dL_dt = sp.Symbol('dL_dt', real=True)  # ∂L/∂t
dH_dq = sp.Symbol('dH_dq', real=True)  # ∂H/∂q_i
dH_dp = sp.Symbol('dH_dp', real=True)  # ∂H/∂p_i
dH_dt = sp.Symbol('dH_dt', real=True)  # ∂H/∂t

# Differential symbols
dq, dp, dqdot, dt_sym = sp.symbols('dq dp dqdot dt', real=True)

# ---------------------------
# Step 1: Lagrangian formulation
# ---------------------------
# L = L(q, q_dot, t) - this is the starting point
print("Step 1: Lagrangian L = L(q, q_dot, t) established")

# ---------------------------
# Step 2: Euler-Lagrange equation
# ---------------------------
# d/dt(∂L/∂q_dot_i) = ∂L/∂q_i
# This is taken from the dependency euler_lagrange_equations
dp_dt = sp.Symbol('dp_dt', real=True)  # d/dt(∂L/∂q_dot) = dp/dt
euler_lagrange_eq = sp.Eq(dp_dt, dL_dq)
print(f"Step 2: Euler-Lagrange equation: {euler_lagrange_eq}")

# ---------------------------
# Step 3: Define conjugate momentum
# ---------------------------
# p_i = ∂L/∂q_dot_i
momentum_def = sp.Eq(p_i, dL_dqdot)
print(f"Step 3: Conjugate momentum definition: {momentum_def}")
# This means dL_dqdot = p_i
assert momentum_def.lhs == p_i, "Step 3 verification failed"

# ---------------------------
# Step 4: Legendre transformation
# ---------------------------
# H(q, p, t) = sum_i p_i * q_dot_i - L(q, q_dot, t)
# For single degree of freedom (sufficient for index-by-index proof):
H_legendre = p_i * q_dot_i - L(q_i, q_dot_i, t)
print(f"Step 4: Hamiltonian via Legendre transform: H = p*q_dot - L")

# ---------------------------
# Step 5: Total differential of H from definition
# ---------------------------
# dH = q_dot * dp + p * d(q_dot) - (∂L/∂q)*dq - (∂L/∂q_dot)*d(q_dot) - (∂L/∂t)*dt
dH_step5 = q_dot_i * dp + p_i * dqdot - dL_dq * dq - dL_dqdot * dqdot - dL_dt * dt_sym
print(f"Step 5: dH from definition = {dH_step5}")

# ---------------------------
# Step 6: Substitute p_i = ∂L/∂q_dot_i
# ---------------------------
# The term p_i * d(q_dot) and -dL_dqdot * d(q_dot) should cancel
# since p_i = dL_dqdot
dH_step6 = dH_step5.subs(dL_dqdot, p_i)
print(f"Step 6: After substitution p = ∂L/∂q_dot: dH = {dH_step6}")

# ---------------------------
# Step 7: Simplify after cancellation
# ---------------------------
# p_i * dqdot - p_i * dqdot = 0
dH_step7 = sp.simplify(dH_step6)
expected_dH_step7 = q_dot_i * dp - dL_dq * dq - dL_dt * dt_sym
assert sp.simplify(dH_step7 - expected_dH_step7) == 0, "Step 7 verification failed"
print(f"Step 7: After cancellation: dH = {expected_dH_step7}")

# ---------------------------
# Step 8: dH from H(q, p, t)
# ---------------------------
# dH = (∂H/∂q)*dq + (∂H/∂p)*dp + (∂H/∂t)*dt
dH_step8 = dH_dq * dq + dH_dp * dp + dH_dt * dt_sym
print(f"Step 8: dH from partials = {dH_step8}")

# ---------------------------
# Step 9: Compare coefficients of dp
# ---------------------------
# From step 7: coefficient of dp is q_dot_i
# From step 8: coefficient of dp is dH_dp
# Therefore: dH_dp = q_dot_i = dq/dt
coeff_dp_step7 = expected_dH_step7.coeff(dp)
coeff_dp_step8 = dH_step8.coeff(dp)
assert coeff_dp_step7 == q_dot_i, "Step 9: coefficient extraction failed"
assert coeff_dp_step8 == dH_dp, "Step 9: coefficient extraction failed"
first_hamilton = sp.Eq(dH_dp, q_dot_i)
print(f"Step 9: First Hamilton equation: ∂H/∂p = q_dot = dq/dt (eq1 proven)")

# ---------------------------
# Step 10: Compare coefficients of dq
# ---------------------------
# From step 7: coefficient of dq is -dL_dq
# From step 8: coefficient of dq is dH_dq
# Therefore: dH_dq = -dL_dq
coeff_dq_step7 = expected_dH_step7.coeff(dq)
coeff_dq_step8 = dH_step8.coeff(dq)
assert coeff_dq_step7 == -dL_dq, "Step 10: coefficient extraction failed"
assert coeff_dq_step8 == dH_dq, "Step 10: coefficient extraction failed"
relation_step10 = sp.Eq(dH_dq, -dL_dq)
print(f"Step 10: Relation ∂H/∂q = -∂L/∂q established: {relation_step10}")

# ---------------------------
# Step 11: Apply Euler-Lagrange equation
# ---------------------------
# From step 2: dp/dt = ∂L/∂q, i.e., dL_dq = dp_dt
# This connects the Lagrangian derivative to momentum time derivative
print(f"Step 11: From Euler-Lagrange: dp/dt = ∂L/∂q, so dL_dq = dp_dt")

# ---------------------------
# Step 12: Derive second Hamilton equation
# ---------------------------
# From step 10: dH_dq = -dL_dq
# From step 11: dL_dq = dp_dt
# Substituting: dH_dq = -dp_dt
# Therefore: dp_dt = -dH_dq, i.e., dp/dt = -∂H/∂q
dH_dq_substituted = (-dL_dq).subs(dL_dq, dp_dt)
assert dH_dq_substituted == -dp_dt, "Step 12: substitution failed"
second_hamilton = sp.Eq(dp_dt, -dH_dq)
print(f"Step 12: Second Hamilton equation: dp/dt = -∂H/∂q (eq2 proven)")

# =====================================================
# Final verification summary
# =====================================================
print("\n=== Final Verification ===")
print(f"eq1 (First Hamilton equation): dq/dt = ∂H/∂p")
print(f"eq2 (Second Hamilton equation): dp/dt = -∂H/∂q")
print("\n=== All derivation steps verified ===")


## Source

📖 **View this entry:** [theoria-dataset.org/entries.html?entry=hamiltons_equations.json](https://theoria-dataset.org/entries.html?entry=hamiltons_equations.json)

This verification code is part of the [THEORIA dataset](https://github.com/theoria-dataset/theoria-dataset), a curated collection of theoretical physics derivations with programmatic verification.

**License:** CC-BY 4.0